# Find research data and make a first prediction

Install welt-client with its parquet extra, then connect your workspace in the first Python cell. Run the cells in order. These are real hosted requests: fitting retains a private training dataset and predictor. No extra helper file is needed. Outputs are deliberately not saved in this source.

In [ ]:
from welt import Client

with Client() as client:
    client.connect()


In [ ]:
from welt import Client
with Client() as client:
    page = client.research_datasets(q="iris", availability="available", limit=10)
print([(item["id"], item["name"]) for item in page["items"]])

Search results may change. We deliberately select the reviewed Iris asset below, not the first search result. The exact content version pins the bytes; the SDK verifies its SHA and size. This pinned version is the canonical table with target `__target__#0`; it stays downloadable through the detail's `previous_content` if a restored-header version becomes current. Then follow all six modelling steps.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from welt import Client, Classifier

# 1. Get a real table from Welt's research inventory.
dataset_id = "asset-d839644b6de36fc5cafdda18c5daeca3"
version = "d839644b6de36fc5cafdda18c5daeca3284a5ce8bb6169aff2fcb3fbb4a12021"
path = Path(mkdtemp(prefix="welt-iris-")) / "iris.parquet"
with Client() as client:
    info = client.research_dataset(dataset_id)
    print("Dataset:", info["name"], "— License:", info["license"])
    for notice in info["license_notices"]:
        print(notice)
    client.download_research_dataset(dataset_id, path, version=version)
data = pd.read_parquet(path)
target = "__target__#0"  # The target recorded in this research snapshot.



In [ ]:
# 2. Look at the rows before modelling.
print(data.head())
print("Rows and columns:", data.shape)
data.info()
print(data[target].value_counts())



In [ ]:
# 3. Set aside test rows before fitting; never train on their answers.
train, test = train_test_split(
    data, test_size=0.2, random_state=9, stratify=data[target])
query = test.drop(columns=target)
y_test = test[target]
print("Training rows:", len(train), "— Test rows:", len(test))



In [ ]:
# 4. Create a model and train it on the training table.
model = Classifier(model="tabicl-v2", random_state=9)
model.fit(train, target=target)



In [ ]:
# 5. Predict the held-out rows, without passing their target column.
predictions = model.predict(query)
print(pd.DataFrame({"actual": y_test.to_numpy(), "predicted": predictions}).head())



In [ ]:
# 6. Compare predictions with the answers we kept aside.
print("Test accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions, zero_division=0))
print("Keep this predictor ID:", model.predictor_id_)


Optional: load the same table with its catalogue metadata. `load_research_dataset` downloads the same verified bytes into a private temporary folder and returns a DataFrame; with `metadata=True` it also returns the licence, schema and column descriptions. It needs pandas and pyarrow (`welt-client[research]`; the `parquet` extra also provides pyarrow). For notice-bearing ZIP content it warns that `license_text` and `attribution_text` apply. `download_research_dataset(..., metadata=True)` instead writes `<path>.metadata.json` beside a saved file.

In [ ]:
with Client() as client:
    loaded, meta = client.load_research_dataset(dataset_id, version=version, metadata=True)
print(meta.name, "— version", meta.version[:12])
print(meta.column_descriptions)
print("Same table as the download:", loaded.equals(data))